In [1]:
import os
from pathlib import Path

cwd = Path.cwd()

if cwd.name == "notebooks":
    os.chdir(cwd.parent)

In [2]:
from torch_geometric.loader import DataLoader

from gjepa.datasets.cv_vis.tmqmg_star import TMQMGStarDataset

/home/katsiaryna/Projects/GraphJEPA/graph-jepa/.venv/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
def eda(ds: TMQMGStarDataset) -> None:
    pos_tensors = [d.pos for d in ds if d.pos is not None]
    z_tensors = [d.z for d in ds if d.z is not None]
    y_tensors = [d.y for d in ds if hasattr(d, "y") and d.y is not None]

    num_atoms = [p.size(0) for p in pos_tensors]
    print(f"Total molecules: {len(ds)}")
    print(f"Average number of atoms: {sum(num_atoms) / len(num_atoms):.2f}")
    print(f"pos[0].shape = {pos_tensors[0].shape}")
    print(f"z[0].shape = {z_tensors[0].shape}")

    if y_tensors:
        print(f"y[0].shape = {y_tensors[0].shape}")
    else:
        print("No target tensors (`y`) found in dataset.")

## DATASET LOADING

In [4]:
for block_3_only in [
    True
]:
    header = " ".join((
        "="*20, f"block_3_only={block_3_only}", "="*20
    ))

    print(header)
    print()

    ds = TMQMGStarDataset(
        root="data/datasets/TMQMG_star",
        block_3_only=block_3_only,
        prediction_type="vector",
        prediction_params={"range": (380, 750)},
        vis_range = (380, 750),
        max_states = 10
    )

    print("Dataset info:")
    eda(ds)

    print()
    print("Batch info:")

    loader = DataLoader(ds, batch_size=32, shuffle=True)

    batch = next(iter(loader))

    print(batch)

    print(f"pos shape: {batch.pos.shape if hasattr(batch, 'pos') else 'N/A'}")
    print(f"z shape: {batch.z.shape if hasattr(batch, 'z') else 'N/A'}")
    if hasattr(batch, "y") and batch.y is not None:
        print(f"y shape: {batch.y.shape}")
    else:
        print("No target tensors (`y`) found in this batch.")
    print("Batch indices:", batch.batch.shape)

    print()
    print("="*len(header))


==================== block_3_only=True ====================



Processing...


Merged dataset: 25898 rows (from 40111 base and 74281 star)


Processing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25898/25898 [00:11<00:00, 2319.09it/s]

Dataset info:
Total molecules: 320
Average number of atoms: 59.89
pos[0].shape = torch.Size([35, 3])
z[0].shape = torch.Size([35])
y[0].shape = torch.Size([1, 370])

Batch info:
DataBatch(y=[32, 370], pos=[1821, 3], z=[1821], smiles=[32], origin_id=[32], CSD_code=[32], batch=[1821], ptr=[33])
pos shape: torch.Size([1821, 3])
z shape: torch.Size([1821])
y shape: torch.Size([32, 370])
Batch indices: torch.Size([1821])




Done!


In [ ]:
for block_3_only in [
    True
]:
    header = " ".join((
        "="*20, f"block_3_only={block_3_only}", "="*20
    ))

    print(header)
    print()

    ds = TMQMGStarDataset(
        root="data/datasets/TMQMG_star",
        block_3_only=block_3_only,
        prediction_type="vector",
        prediction_params={"range": (380, 750)},
        vis_range = (380, 750),
        max_states = 4
    )

    print("Dataset info:")
    eda(ds)

    print()
    print("Batch info:")

    loader = DataLoader(ds, batch_size=32, shuffle=True)

    batch = next(iter(loader))

    print(batch)

    print(f"pos shape: {batch.pos.shape if hasattr(batch, 'pos') else 'N/A'}")
    print(f"z shape: {batch.z.shape if hasattr(batch, 'z') else 'N/A'}")
    if hasattr(batch, "y") and batch.y is not None:
        print(f"y shape: {batch.y.shape}")
    else:
        print("No target tensors (`y`) found in this batch.")
    print("Batch indices:", batch.batch.shape)

    print()
    print("="*len(header))


Processing...


==================== block_3_only=True ====================



In [ ]:
for block_3_only in [
    True
]:
    header = " ".join((
        "="*20, f"block_3_only={block_3_only}", "="*20
    ))

    print(header)
    print()

    ds = TMQMGStarDataset(
        root="data/datasets/TMQMG_star",
        block_3_only=block_3_only,
        prediction_type="vector",
        prediction_params={"range": (380, 750)},
        vis_range = (0, 2000),
        max_states = 10
    )

    print("Dataset info:")
    eda(ds)

    print()
    print("Batch info:")

    loader = DataLoader(ds, batch_size=32, shuffle=True)

    batch = next(iter(loader))

    print(batch)

    print(f"pos shape: {batch.pos.shape if hasattr(batch, 'pos') else 'N/A'}")
    print(f"z shape: {batch.z.shape if hasattr(batch, 'z') else 'N/A'}")
    if hasattr(batch, "y") and batch.y is not None:
        print(f"y shape: {batch.y.shape}")
    else:
        print("No target tensors (`y`) found in this batch.")
    print("Batch indices:", batch.batch.shape)

    print()
    print("="*len(header))


## TESTING OUTPUT FUNCTIONS

In [5]:
import torch
from typing import List, Tuple
import numpy as np
import matplotlib.pyplot as pl
def _build_top_pairs(transitions: List[Tuple[float, float]], num_pairs: int = 20) -> torch.Tensor:
    if not transitions:
        return torch.zeros(1, num_pairs * 2, dtype=torch.float32)

    selected = transitions[:num_pairs]
    vec = []
    for lam, f in selected:
        vec.extend([lam, f])
    while len(vec) < num_pairs * 2:
        vec.extend([0.0, 0.0])
    return torch.tensor([vec], dtype=torch.float32)


row_transitions = [
    (306.8, 0.0028),
    (285.19, 0.1514),
    (273.68, 0.0016),
    (257.5, 0.2278),
    (251.04, 0.0491),
    (250.78, 0.0066),
    (238.1, 0.0102),
    (237.77, 0.0147),
    (232.84, 0.0359),
    (227.57, 0.0012),
    # (222.31, 0.0),   # 11th → ignored
    # ... etc.
]

result = _build_top_pairs(row_transitions, num_pairs=10)
print(result)
print(result.shape)

expected = torch.tensor([[
    306.8,   0.0028,
    285.19,  0.1514,
    273.68,  0.0016,
    257.5,   0.2278,
    251.04,  0.0491,
    250.78,  0.0066,
    238.1,   0.0102,
    237.77,  0.0147,
    232.84,  0.0359,
    227.57,  0.0012
]], dtype=torch.float32)

assert torch.allclose(result, expected), "Test failed!"
print("Test passed!")

tensor([[3.0680e+02, 2.8000e-03, 2.8519e+02, 1.5140e-01, 2.7368e+02, 1.6000e-03,
         2.5750e+02, 2.2780e-01, 2.5104e+02, 4.9100e-02, 2.5078e+02, 6.6000e-03,
         2.3810e+02, 1.0200e-02, 2.3777e+02, 1.4700e-02, 2.3284e+02, 3.5900e-02,
         2.2757e+02, 1.2000e-03]])
torch.Size([1, 20])
Test passed!


In [6]:
def _build_absorption_vector(
    transitions: List[Tuple[float, float]],
    wavelength_range: Tuple[float, float] = (300.0, 700.0),
    num_bins: int = 400
) -> torch.Tensor:
    start, end = wavelength_range
    hist = np.zeros(num_bins)

    for lam, f in transitions:
        if start <= lam <= end:
            idx = int((lam - start) / (end - start) * num_bins)
            idx = min(max(idx, 0), num_bins - 1)
            hist[idx] += f

    return torch.tensor(hist, dtype=torch.float32).unsqueeze(0)

vec = _build_absorption_vector(row_transitions, wavelength_range=(300.0, 700.0), num_bins=400)
print(f"Shape: {vec.shape}")

wavelengths = np.linspace(300.0, 700.0, 400)
intensity = vec.squeeze(0).numpy()

plt.figure(figsize=(10, 5))
plt.plot(wavelengths, intensity, color='purple', linewidth=1.2)
plt.fill_between(wavelengths, intensity, alpha=0.3, color='purple')
plt.xlabel("Wavelength (nm)")
plt.ylabel("Oscillator Strength (f)")
plt.title("Discrete Absorption Spectrum (Histogram Bins)")
plt.grid(True, alpha=0.3)
plt.xlim(300, 700)
plt.ylim(0, None)
plt.tight_layout()
plt.show()

Shape: torch.Size([1, 400])


NameError: name 'plt' is not defined

## TODO: prepare a Hydra config

In [14]:
# from pathlib import Path
# from typing import Any, Literal, Optional, Sequence, Union

# import yaml
# from pydantic import BaseModel, Extra, Field

# class TMQMDatasetConfig(BaseModel, extra="forbid"):
#     name: str
#     root_dir: Path
#     in_channels: int

#     prediction_type: Optional[Literal["pairs", "vector"]] = None
#     prediction_params: Optional[dict[str, Any]] = None
#     task_type: Optional[str] = None

#     main_metric: Optional[str] = None
#     metric_mode: Optional[str] = None

#     split_ratios: Optional[Sequence[float]] = None
#     vis_range: Optional[Sequence[float]] = None
#     lambda_: Optional[float] = Field(None, alias="lambda")

#     pre_transforms: dict[str, Any] = Field(default_factory=dict)
#     transforms: dict[str, Any] = Field(default_factory=dict)

#     subset: Optional[str] = None

#     @property
#     def cutoff(self) -> float:
#         return self.transforms.get("AddEdgesAndDistances", {}).get("cutoff", 5.0)


# def load_configs(path: Path) -> list[TMQMDatasetConfig]:
#     raw = yaml.safe_load(path.read_text())
#     items = raw if isinstance(raw, list) else [raw]
#     return [TMQMDatasetConfig(**item) for item in items]

# load_configs(Path("config/dataset/TMQM_SPECTO_VECTOR_CONTINUOUS.yaml"))

[TMQMDatasetConfig(name='TMQM_SPECTO', root_dir=PosixPath('data/datasets/TMQM_SPECTO'), in_channels=1, prediction_type='vector', prediction_params={'range': '(280, 350)', 'approximation': 'gaussian'}, task_type='regression', main_metric='MAE', metric_mode='min', split_ratios=[0.8, 0.1], vis_range=[400.0, 700.0], lambda_=10.0, pre_transforms={}, transforms={'AddEdgesAndDistances': {'cutoff': 5.0}}, subset='cv_vis')]